In [ ]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes huggingface_hub pandas scikit-learn

In [ ]:
import gc
import json
import ast
import os
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_from_disk
from huggingface_hub import login
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

PREV_OUTPUT_DIR = Path("/kaggle/input/notebooks/ahmedsamirafattah/toolforge-fine-tuning-qwen2-5-for-reliable-tool-c/toolforge")
TEST_SPLIT_DIR  = PREV_OUTPUT_DIR / "data/processed_splits/test"
BASE_CSV        = PREV_OUTPUT_DIR / "results/base_predictions.csv"

RESULTS_DIR = Path("/kaggle/working/toolforge/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── model ids ──────────────────────────────────────────────────────────────
BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_REPO  = "Ahmed-Samir-Abdel-fattah/toolforge-qwen2.5-1.5b-lora"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace Hub.")

In [ ]:
test_ds = load_from_disk(str(TEST_SPLIT_DIR))
print(f"Test split loaded: {len(test_ds)} samples")
print("Columns:", test_ds.column_names)

# Sanity check — show one sample
sample = test_ds[0]
print("\n--- Sample prompt (first 400 chars) ---")
print(sample["prompt"][:400])
print("\n--- Reference assistant output ---")
print(sample["assistant"][:400])

In [ ]:
def make_json_safe(obj: Any) -> Any:
    if obj is ...:
        return None
    if isinstance(obj, set):
        return list(obj)
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(x) for x in obj]
    try:
        json.dumps(obj)
        return obj
    except TypeError:
        return str(obj)

In [ ]:
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL | re.IGNORECASE)


def parse_json_object(text: str) -> Optional[Any]:
    text = text.strip()
    candidates = [text]

    first_brace = text.find("{")
    last_brace = text.rfind("}")
    if first_brace >= 0 and last_brace > first_brace:
        candidates.append(text[first_brace : last_brace + 1])

    first_bracket = text.find("[")
    last_bracket = text.rfind("]")
    if first_bracket >= 0 and last_bracket > first_bracket:
        candidates.append(text[first_bracket : last_bracket + 1])

    for candidate in candidates:
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(candidate)
            except Exception:
                pass
    return None


def normalize_call(obj: Any) -> Optional[Dict[str, Any]]:
    if not isinstance(obj, dict):
        return None

    name = obj.get("name") or obj.get("function") or obj.get("tool") or obj.get("tool_name")
    arguments = obj.get("arguments")
    if arguments is None:
        arguments = obj.get("parameters") or obj.get("args") or {}

    if isinstance(name, dict):
        arguments = name.get("arguments") or arguments
        name = name.get("name")

    if isinstance(arguments, str):
        parsed_args = parse_json_object(arguments)
        if parsed_args is not None:
            arguments = parsed_args

    if not name:
        return None

    return {"name": str(name), "arguments": arguments if arguments is not None else {}}


def parse_tool_calls(text: str) -> Tuple[List[Dict[str, Any]], bool]:
    if not isinstance(text, str):
        return [], False

    chunks = TOOL_CALL_RE.findall(text)
    if not chunks:
        parsed = parse_json_object(text)
        chunks = []
        if isinstance(parsed, list):
            calls = [normalize_call(item) for item in parsed]
            calls = [call for call in calls if call is not None]
            return calls, bool(calls)
        call = normalize_call(parsed)
        return ([call], True) if call else ([], False)

    calls = []
    all_valid = True
    for chunk in chunks:
        parsed = parse_json_object(chunk)
        call = normalize_call(parsed)
        if call is None:
            all_valid = False
        else:
            calls.append(call)
    return calls, all_valid and bool(calls)


def flatten_args(value: Any, prefix: str = "") -> List[str]:
    items = []
    if isinstance(value, dict):
        for key in sorted(value.keys()):
            next_prefix = f"{prefix}.{key}" if prefix else str(key)
            items.extend(flatten_args(value[key], next_prefix))
    elif isinstance(value, list):
        for idx, item in enumerate(value):
            items.extend(flatten_args(item, f"{prefix}[{idx}]"))
    else:
        items.append(f"{prefix}={value}")
    return items


def argument_f1(pred_calls: List[Dict[str, Any]], ref_calls: List[Dict[str, Any]]) -> float:
    pred_items = []
    ref_items = []

    for call in pred_calls:
        pred_items.extend([f"{call['name']}:{x}" for x in flatten_args(call.get("arguments", {}))])
    for call in ref_calls:
        ref_items.extend([f"{call['name']}:{x}" for x in flatten_args(call.get("arguments", {}))])

    pred_set = set(pred_items)
    ref_set = set(ref_items)

    if not pred_set and not ref_set:
        return 1.0
    if not pred_set or not ref_set:
        return 0.0

    tp = len(pred_set & ref_set)
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(ref_set) if ref_set else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def extract_available_tool_names(tools_text: str) -> set:
    parsed = parse_json_object(tools_text)
    names = set()

    def walk(obj: Any):
        if isinstance(obj, dict):
            possible = obj.get("name") or obj.get("function", {}).get("name") if isinstance(obj.get("function"), dict) else obj.get("name")
            if possible:
                names.add(str(possible))
            for value in obj.values():
                walk(value)
        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    walk(parsed)
    return names


def score_prediction(prediction: str, reference: str, tools_text: str) -> Dict[str, Any]:
    pred_calls, pred_valid = parse_tool_calls(prediction)
    ref_calls, ref_valid = parse_tool_calls(reference)

    pred_names = [call["name"] for call in pred_calls]
    ref_names = [call["name"] for call in ref_calls]
    available_names = extract_available_tool_names(tools_text)

    no_call_match = not pred_calls and not ref_calls
    function_correct = pred_names == ref_names
    hallucinated = bool(available_names) and any(name not in available_names for name in pred_names)

    return {
        "format_valid": bool(pred_valid or no_call_match),
        "function_correct": bool(function_correct),
        "argument_f1": argument_f1(pred_calls, ref_calls),
        "hallucinated": bool(hallucinated),
        "pred_calls": make_json_safe(pred_calls),
        "ref_calls": make_json_safe(ref_calls),
    }
print("Scoring functions defined.")

# Quick sanity test
_pred = '<tool_call>{"name": "get_weather", "arguments": {"city": "Cairo"}}</tool_call>'
_ref  = '<tool_call>{"name": "get_weather", "arguments": {"city": "Cairo"}}</tool_call>'
_tools = json.dumps([{"type": "function", "function": {"name": "get_weather"}}])
_s = score_prediction(_pred, _ref, _tools)
assert _s["format_valid"]     == True
assert _s["function_correct"] == True
assert _s["argument_f1"]      == 1.0
assert _s["hallucinated"]     == False
print("Sanity check passed ✓")

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model.eval()

print("Loading LoRA adapter from HuggingFace Hub...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
ft_model.eval()

print("Fine-tuned model ready.")
print(f"Device: {next(ft_model.parameters()).device}")

In [ ]:
def generate_response(model, prompt: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_model(model, dataset: Dataset, label: str, limit: Optional[int] = None) -> Tuple[pd.DataFrame, Dict[str, float]]:
    rows = []
    eval_count = len(dataset) if limit is None else min(limit, len(dataset))

    for idx in range(eval_count):
        sample = dataset[idx]
        prediction = generate_response(model, sample["prompt"])
        scores = score_prediction(prediction, sample["assistant"], sample.get("tools", "[]"))
        rows.append({
            "model": label,
            "index": idx,
            "prompt": sample["prompt"],
            "reference": sample["assistant"],
            "prediction": prediction,
            "format_valid": scores["format_valid"],
            "function_correct": scores["function_correct"],
            "argument_f1": scores["argument_f1"],
            "hallucinated": scores["hallucinated"],
            "pred_calls": json.dumps(make_json_safe(scores["pred_calls"]), ensure_ascii=False),
            "ref_calls":  json.dumps(make_json_safe(scores["ref_calls"]),  ensure_ascii=False),
        })

    df = pd.DataFrame(rows)
    summary = {
        "model": label,
        "n": len(df),
        "format_accuracy": float(df["format_valid"].mean()),
        "function_accuracy": float(df["function_correct"].mean()),
        "argument_f1": float(df["argument_f1"].mean()),
        "hallucination_rate": float(df["hallucinated"].mean()),
    }
    return df, summary


print("Evaluation functions defined.")

In [ ]:
ft_eval_df, ft_summary = evaluate_model(
    ft_model,
    test_ds,
    label="ToolForge (QLoRA)",
    limit=200,       

)

ft_eval_df.to_csv(RESULTS_DIR / "finetuned_predictions.csv", index=False)
print(f"\nSaved to {RESULTS_DIR / 'finetuned_predictions.csv'}")
print("\nFine-tuned model summary:")
print(pd.DataFrame([ft_summary]).to_string(index=False))

In [ ]:
base_df = pd.read_csv(BASE_CSV)

base_summary = {
    "model":              "Base Qwen2.5-1.5B",
    "n":                  len(base_df),
    "format_accuracy":    float(base_df["format_valid"].mean()),
    "function_accuracy":  float(base_df["function_correct"].mean()),
    "argument_f1":        float(base_df["argument_f1"].mean()),
    "hallucination_rate": float(base_df["hallucinated"].mean()),
}

cmp = pd.DataFrame([base_summary, ft_summary])

deltas = {
    "model":              "Δ (fine-tuned − base)",
    "n":                  "-",
    "format_accuracy":    ft_summary["format_accuracy"]    - base_summary["format_accuracy"],
    "function_accuracy":  ft_summary["function_accuracy"]  - base_summary["function_accuracy"],
    "argument_f1":        ft_summary["argument_f1"]        - base_summary["argument_f1"],
    # hallucination: negative delta is good (lower is better)
    "hallucination_rate": ft_summary["hallucination_rate"] - base_summary["hallucination_rate"],
}
cmp = pd.concat([cmp, pd.DataFrame([deltas])], ignore_index=True)

cmp.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)

print("=" * 75)
print("TOOLFORGE — BASE VS FINE-TUNED COMPARISON")
print("=" * 75)

display_rows = []
for _, row in cmp.iterrows():
    if row["model"] == "Δ (fine-tuned − base)":
        display_rows.append({
            "Model":             row["model"],
            "Format Accuracy":   f"{row['format_accuracy']:+.1%}",
            "Function Accuracy": f"{row['function_accuracy']:+.1%}",
            "Argument F1":       f"{row['argument_f1']:+.3f}",
            "Hallucination Rate":f"{row['hallucination_rate']:+.1%}",
        })
    else:
        display_rows.append({
            "Model":             row["model"],
            "Format Accuracy":   f"{row['format_accuracy']:.1%}",
            "Function Accuracy": f"{row['function_accuracy']:.1%}",
            "Argument F1":       f"{row['argument_f1']:.3f}",
            "Hallucination Rate":f"{row['hallucination_rate']:.1%}",
        })

print(pd.DataFrame(display_rows).to_string(index=False))
print("=" * 75)
print("(hallucination rate: lower is better; all other metrics: higher is better)")